# Choosing a partner without knowing strength in advance (calibration-sample simulation)

Follows `04_results/partner_choice_calibration_plan.json`. For each deployed model, a partner of a different architecture family is chosen by one of several rules, using either a small graded **calibration sample** of local images or **no labels at all**; the chosen partner is scored on the remaining images by net severe eyes rescued at matched referral volume.

Sites: EyePACS confirmation set (22 full-recipe zoo models, 25% referral) and DDR (11 paper-one architectures, 55.5% referral). CPU; about 15 to 25 minutes.

In [2]:
# Cell 1: load
from google.colab import drive
drive.mount('/content/drive')
import os, json, datetime, itertools, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score
BASE = '/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2'
SPLITS, PREDS, RESULTS, FULL = f'{BASE}/01_splits', f'{BASE}/03_predictions', f'{BASE}/04_results', f'{BASE}/04_results/ballistics_full'
SIZES = [50, 100, 200, 500, 1000, 2000]; DRAWS = 200; RNG = np.random.default_rng(20260926)
def pick(df, c): return next(x for x in c if x in df.columns)
def load_scores(path, ids):
    df = pd.read_csv(path, dtype={'image_id': str}); pc = pick(df, ['predicted_probability','probability','prob'])
    s = df.drop_duplicates('image_id').set_index('image_id')[pc].astype(float).reindex(ids); assert s.notna().all(), path; return s.to_numpy()
# EyePACS: full-recipe zoo
fr = pd.read_csv(f'{SPLITS}/eyepacs_split_fresh_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
Z = pd.DataFrame([('resnet50','resnet'),('resnet50_swsl','resnet'),('resnext50','resnet'),('regnety_032','regnet'),('densenet121','densenet'),
    ('efficientnet_b3','efficientnet'),('effnetv2_s','efficientnet'),('mobilenetv3','mobilenet'),('convnext_t','convnext'),('convnext_t22k','convnext'),
    ('inception_v3','inception'),('vgg16_bn','vgg'),('vit_b','vit'),('vit_b_mae','vit'),('vit_b_dinov2','vit'),('vit_b_clip','vit'),('vit_s','vit'),
    ('deit_s','deit'),('swin_t','swin'),('beitv2_b','beit'),('mixer_b16','mixer'),('retfound','retfound')], columns=['name','family']).set_index('name')
SITES = {'EyePACS': {'ids': fr.image_id.to_numpy(), 'grade': fr[pick(fr, ['level','grade'])].to_numpy().astype(int), 'cluster': fr.patient_id.astype(str).to_numpy(), 'f': 0.25,
                     'family': Z.family.to_dict(), 'S': {m: load_scores(f'{FULL}/preds/{m}_fresh.csv', fr.image_id.to_numpy()) for m in Z.index}}}
dd = pd.read_csv(f'{SPLITS}/ddr_split_intraset_v2.csv', dtype={'image_id': str}).sort_values('image_id').reset_index(drop=True)
DDR_MODELS = {'densenet121_binary':'densenet','densenet121_5class':'densenet','efficientnet_b3_binary':'efficientnet','efficientnet_b3_5class':'efficientnet',
              'resnet50_binary':'resnet','resnet50_5class':'resnet','vit_base_binary':'vit','vit_base_5class':'vit','retfound_binary':'retfound','retfound_5class':'retfound','retfound_adversarial':'retfound'}
A = (dd.half == 'A').to_numpy()
SITES['DDR'] = {'ids': dd.image_id.to_numpy(), 'grade': dd.grade.to_numpy().astype(int), 'cluster': dd.group.astype(str).to_numpy(), 'f': 1.31 * float((dd.grade[A] >= 2).mean()),
                'sample_unit': 'image',   # DDR filename groups are grade-clustered (consecutive numbering), so group sampling gives unrepresentative calibration samples
                'family': DDR_MODELS, 'S': {m: load_scores(f'{PREDS}/ddr_predictions_{m}_v2.csv', dd.image_id.to_numpy()) for m in DDR_MODELS}}
PT = pd.read_csv(f'{FULL}/pair_table.csv'); DEG = {}
for _, r in PT.iterrows(): DEG[(r.a, r.b)] = DEG[(r.b, r.a)] = float(r.sim_degradation)
for k, S in SITES.items(): print(k, len(S['ids']), 'eyes |', len(S['S']), 'models | f =', round(S['f'], 3), '| severe eyes', int((S['grade'] >= 3).sum()))

Mounted at /content/drive
EyePACS 15794 eyes | 22 models | f = 0.25 | severe eyes 712
DDR 12422 eyes | 11 models | f = 0.555 | severe eyes 1149


In [3]:
# Cell 2: per-eye contributions for every ordered pair (thresholds on the full set), strength, consensus
def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind='stable')[:int(round(f * len(s)))]] = True; return r
for k, S in SITES.items():
    g = S['grade']; sev = g >= 3; M = list(S['S']); S['models'] = M; f = S['f']
    R = {m: refer(S['S'][m], f) for m in M}; RK = {m: rankdata(S['S'][m]) / len(g) for m in M}
    S['net'] = {}
    for a in M:
        for b in M:
            if a == b or S['family'][a] == S['family'][b]: continue
            e = refer(np.maximum(RK[a], RK[b]), f)
            S['net'][(a, b)] = (sev & ~R[a]).astype(np.int8) - (sev & ~e).astype(np.int8)     # per-eye net severe rescue
    S['sevmiss_full'] = {m: int((sev & ~R[m]).sum()) for m in M}; S['R'] = R
    C = np.array([[spearmanr(S['S'][a], S['S'][b])[0] if a != b else np.nan for b in M] for a in M])
    S['consensus'] = dict(zip(M, np.nanmean(C, axis=1)))
    true_strength = [-S['sevmiss_full'][m] for m in M]
    S['consensus_vs_strength'] = round(float(spearmanr([S['consensus'][m] for m in M], true_strength)[0]), 3)
    print(k, 'consensus-vs-true-strength rank correlation:', S['consensus_vs_strength'])

EyePACS consensus-vs-true-strength rank correlation: 0.489
DDR consensus-vs-true-strength rank correlation: 0.264


In [4]:
# Cell 3: simulation
def eligible(S, a): return [b for b in S['models'] if b != a and S['family'][b] != S['family'][a]]
def choose(S, a, rule, sample_mask):
    el = eligible(S, a); g = S['grade']
    if rule == 'sample_auc':
        ref = g[sample_mask] >= 2
        if ref.all() or (~ref).all(): return None
        return max(el, key=lambda b: roc_auc_score(ref, S['S'][b][sample_mask]))
    if rule == 'sample_severe_misses':
        sev = (g >= 3) & sample_mask
        return min(el, key=lambda b: ((sev & ~S['R'][b]).sum(), -S['consensus'][b]))
    if rule == 'consensus': return max(el, key=lambda b: S['consensus'][b])
    if rule == 'consensus_plus_degradation':
        top = sorted(el, key=lambda b: -S['consensus'][b])[:max(1, len(el) // 2)]
        return min(top, key=lambda b: DEG.get((a, b), np.inf))
    if rule == 'true_strongest': return min(el, key=lambda b: S['sevmiss_full'][b])
def draw_sample(S, n):
    if S.get('sample_unit') == 'image':
        mask = np.zeros(len(S['grade']), bool); mask[RNG.choice(len(mask), n, replace=False)] = True; return mask
    cl = S['cluster']; u = RNG.permutation(np.unique(cl)); mask = np.zeros(len(cl), bool); cnt = 0
    order = {c: i for i, c in enumerate(u)}; sizes = pd.Series(cl).value_counts()
    chosen = []
    for c in u:
        chosen.append(c); cnt += sizes[c]
        if cnt >= n: break
    return np.isin(cl, chosen)
RES = {}
for k, S in SITES.items():
    rules = ['sample_auc', 'sample_severe_misses', 'consensus'] + (['consensus_plus_degradation'] if k == 'EyePACS' else [])
    fixed = {r: {a: choose(S, a, r, None) for a in S['models']} for r in rules if r.startswith('consensus')}
    fixed['true_strongest'] = {a: choose(S, a, 'true_strongest', None) for a in S['models']}
    out = {n: {r: [] for r in rules + ['random', 'true_strongest', 'oracle']} for n in SIZES}
    hits = {n: {r: [] for r in ('sample_auc', 'sample_severe_misses')} for n in SIZES}
    for n in SIZES:
        for d in range(DRAWS):
            m = draw_sample(S, n); ev = ~m
            for r in rules + ['random', 'true_strongest', 'oracle']:
                vals = []
                for a in S['models']:
                    el = eligible(S, a); score = {b: int(S['net'][(a, b)][ev].sum()) for b in el}
                    if r == 'random': vals.append(np.mean(list(score.values())))
                    elif r == 'oracle': vals.append(max(score.values()))
                    elif r in fixed: vals.append(score[fixed[r][a]])
                    else:
                        b = choose(S, a, r, m)
                        if b is None: continue
                        vals.append(score[b])
                        if r in hits[n]: hits[n][r].append(b == fixed['true_strongest'][a])
                if vals: out[n][r].append(float(np.mean(vals)))
        print(k, 'n =', n, {r: round(float(np.mean(v)), 2) for r, v in out[n].items()})
    RES[k] = {'out': out, 'hits': hits}

EyePACS n = 50 {'sample_auc': 15.12, 'sample_severe_misses': 13.97, 'consensus': 14.28, 'consensus_plus_degradation': 17.04, 'random': 11.56, 'true_strongest': 17.95, 'oracle': 20.26}
EyePACS n = 100 {'sample_auc': 16.28, 'sample_severe_misses': 13.99, 'consensus': 14.23, 'consensus_plus_degradation': 16.97, 'random': 11.52, 'true_strongest': 17.89, 'oracle': 20.19}
EyePACS n = 200 {'sample_auc': 17.15, 'sample_severe_misses': 13.91, 'consensus': 14.15, 'consensus_plus_degradation': 16.89, 'random': 11.45, 'true_strongest': 17.77, 'oracle': 20.08}
EyePACS n = 500 {'sample_auc': 17.2, 'sample_severe_misses': 13.6, 'consensus': 13.84, 'consensus_plus_degradation': 16.54, 'random': 11.22, 'true_strongest': 17.41, 'oracle': 19.72}
EyePACS n = 1000 {'sample_auc': 16.56, 'sample_severe_misses': 13.79, 'consensus': 13.41, 'consensus_plus_degradation': 16.03, 'random': 10.9, 'true_strongest': 16.86, 'oracle': 19.13}
EyePACS n = 2000 {'sample_auc': 15.86, 'sample_severe_misses': 13.3, 'consensu

In [5]:
# Cell 4: summaries, figure, write
import matplotlib as mpl; mpl.use('Agg'); import matplotlib.pyplot as plt
SUM = {}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
COL = {'random': '#999999', 'sample_auc': '#0072B2', 'sample_severe_misses': '#56B4E9', 'consensus': '#E69F00', 'consensus_plus_degradation': '#CC79A7',
       'true_strongest': '#009E73', 'oracle': '#000000'}
for ax, (k, R_) in zip(axes, RES.items()):
    out = R_['out']; SUM[k] = {}
    for n in SIZES:
        rnd, ts = np.mean(out[n]['random']), np.mean(out[n]['true_strongest']); orc = np.mean(out[n]['oracle'])
        SUM[k][n] = {r: {'mean_net_severe_rescued': round(float(np.mean(v)), 2), 'p5': round(float(np.percentile(v, 5)), 2), 'p95': round(float(np.percentile(v, 95)), 2),
                         'share_of_true_strongest_advantage_over_random': round(float((np.mean(v) - rnd) / (ts - rnd)), 3) if ts != rnd else None,
                         'share_of_oracle_advantage_over_random': round(float((np.mean(v) - rnd) / (orc - rnd)), 3) if orc != rnd else None}
                     for r, v in out[n].items()}
        for r, h in R_['hits'][n].items(): SUM[k][n][r]['picks_true_strongest'] = round(float(np.mean(h)), 3) if h else None
    for r in out[SIZES[0]]:
        y = [np.mean(out[n][r]) for n in SIZES]; lo = [np.percentile(out[n][r], 5) for n in SIZES]; hi = [np.percentile(out[n][r], 95) for n in SIZES]
        ls = '--' if r in ('random', 'true_strongest', 'oracle', 'consensus', 'consensus_plus_degradation') else '-'
        ax.plot(SIZES, y, ls, color=COL[r], lw=2, marker='o' if ls == '-' else None, label=r.replace('_', ' '))
        if ls == '-': ax.fill_between(SIZES, lo, hi, color=COL[r], alpha=0.12)
    ax.set_xscale('log'); ax.set_xlabel('Graded local images in the calibration sample'); ax.set_ylabel('Net severe eyes rescued per deployed model (matched volume)')
    ax.set_title(f"{k}  (consensus vs true strength rho {SITES[k]['consensus_vs_strength']:+.2f})", loc='left', fontsize=10, weight='bold'); ax.spines[['top','right']].set_visible(False)
axes[0].legend(fontsize=7.5, frameon=False)
fig.suptitle('Choosing a partner without knowing strength: how many graded local images are needed?', x=0.01, ha='left', fontsize=12, weight='bold')
fig.tight_layout(); fig.savefig(f'{BASE}/05_figures/partner_choice_calibration_v2.png', dpi=220, bbox_inches='tight'); plt.close(fig)
res = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'plan': 'partner_choice_calibration_plan.json', 'status': 'exploratory',
       'consensus_vs_true_strength': {k: S['consensus_vs_strength'] for k, S in SITES.items()}, 'summary': SUM,
       'deviation_from_plan': 'DDR calibration samples drawn by image, not filename group: groups of consecutively numbered DDR images are grade-clustered, which made group-sampled calibration sets unrepresentative (first run)',
       'n_models': {k: len(S['models']) for k, S in SITES.items()}, 'referral_fraction': {k: round(S['f'], 4) for k, S in SITES.items()}}
json.dump(res, open(f'{RESULTS}/partner_choice_calibration_results.json', 'w'), indent=2, default=str)
for k in SUM:
    print(k); print(pd.DataFrame({n: {r: v['share_of_true_strongest_advantage_over_random'] for r, v in SUM[k][n].items()} for n in SIZES}).to_string())
print('Wrote 04_results/partner_choice_calibration_results.json and 05_figures/partner_choice_calibration_v2.png')

EyePACS
                             50     100    200    500    1000   2000
sample_auc                  0.557  0.747  0.902  0.966  0.950  1.018
sample_severe_misses        0.377  0.388  0.388  0.384  0.485  0.565
consensus                   0.426  0.425  0.427  0.423  0.422  0.420
consensus_plus_degradation  0.858  0.855  0.859  0.858  0.861  0.857
random                      0.000  0.000  0.000  0.000  0.000  0.000
true_strongest              1.000  1.000  1.000  1.000  1.000  1.000
oracle                      1.362  1.361  1.365  1.372  1.380  1.402
DDR
                       50     100    200    500    1000   2000
sample_auc           -0.257 -0.318 -0.369 -0.425 -0.434 -0.506
sample_severe_misses  0.021  0.053  0.127  0.351  0.415  0.576
consensus            -0.020 -0.019 -0.023 -0.021 -0.020 -0.017
random                0.000  0.000  0.000  0.000  0.000  0.000
true_strongest        1.000  1.000  1.000  1.000  1.000  1.000
oracle                1.096  1.095  1.098  1.104  1.106  1